# Task 5: Spark Structured Streaming -> MongoDB

### Mục tiêu
- Sử dụng **Spark Structured Streaming** đọc topic `cpg-metadata` từ Kafka.
- Parse JSON payload theo schema đã định nghĩa.
- Ghi kết quả vào **MongoDB** collection `peft_db.source_metadata` ở chế độ **append**.
- Dùng **checkpoint** để đảm bảo fault-tolerance và exactly-once delivery.

### Thiết kế

| Thành phần | Giá trị |
|:---|:---|
| **Kafka topic** | `cpg-metadata` |
| **MongoDB URI** | `mongodb://127.0.0.1:27017` |
| **Database** | `peft_db` |
| **Collection** | `source_metadata` |
| **Checkpoint** | `checkpoints/task5_metadata/` |
| **Trigger** | `processingTime=10 seconds` |

### Kiến trúc

```text
cpg_parser.py  -->  Kafka (cpg-metadata)  -->  Spark Structured Streaming  -->  MongoDB
                    localhost:9092              pyspark 3.5.0 / 10s trigger      peft_db.source_metadata
                                                       |
                                                       v
                                          checkpoints/task5_metadata/
```


### 1. Khởi động MongoDB (Docker)

In [21]:
import subprocess, sys, time
from pathlib import Path

TASK5_DIR = Path.cwd() / 'src' / 'task5'

print('[INFO] Khởi động MongoDB container...')
r = subprocess.run(
    ['docker', 'compose', 'up', '-d'],
    cwd=str(TASK5_DIR), capture_output=True, text=True
)
print(r.stdout or r.stderr)

# Kiểm tra container
r2 = subprocess.run(
    ['docker', 'ps', '--filter', 'name=mongo', '--format', 'table {{.Names}}\t{{.Status}}'],
    capture_output=True, text=True
)
print(r2.stdout)

[INFO] Khởi động MongoDB container...
time="2026-07-24T13:55:31+07:00" level=warning msg="e:\\BD\\src\\task5\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container cpg-mongodb Running 

NAMES         STATUS
cpg-mongodb   Up 14 minutes



### 2. Chạy Spark Structured Streaming

> ⚠️ **Lưu ý:** `spark-submit` là tiến trình **long-running** (blocking), cần chạy ở **terminal riêng** song song với notebook.
>
> Lệnh cần chạy trong terminal:
> ```cmd
> set HADOOP_HOME=D:\hadoop
> set PATH=D:\hadoop\bin;%PATH%
> set PYSPARK_PYTHON=D:\Anaconda3\envs\min_ds-env\python.exe
> set PYSPARK_DRIVER_PYTHON=D:\Anaconda3\envs\min_ds-env\python.exe
> D:\Anaconda3\envs\min_ds-env\Scripts\spark-submit --packages "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.mongodb.spark:mongo-spark-connector_2.12:10.3.0" src\task5\ingest.py
> ```

In [22]:
import shutil, sys, os
from pathlib import Path

# ── 1. Tìm spark-submit tự động ─────────────────────────
spark_submit = shutil.which('spark-submit')
if spark_submit is None:
    try:
        import pyspark
        _py = Path(sys.executable)
        for c in [_py.parent / 'spark-submit',
                  _py.parent / 'spark-submit.cmd',
                  _py.parent / 'Scripts' / 'spark-submit',
                  _py.parent / 'Scripts' / 'spark-submit.cmd',
                  Path(pyspark.__file__).parent / 'bin' / 'spark-submit']:
            if c.exists():
                spark_submit = str(c)
                break
    except ImportError:
        pass

# ── 2. Tìm HADOOP_HOME tự động ──────────────────────────
hadoop_home = os.environ.get('HADOOP_HOME', '')
if not hadoop_home:
    _wu = shutil.which('winutils') or shutil.which('winutils.exe')
    if _wu:
        hadoop_home = str(Path(_wu).parent.parent)
    else:
        for _d in 'CDEF':
            for _sub in ['hadoop', 'tools\\hadoop', 'dev\\hadoop']:
                _p = Path(f'{_d}:\\{_sub}')
                if (_p / 'bin' / 'winutils.exe').exists():
                    hadoop_home = str(_p); break
            if hadoop_home: break

python_exe = sys.executable

print(f'[OK] Python       : {python_exe}')
print(f'[OK] spark-submit : {spark_submit or "NOT FOUND — pip install pyspark==3.5.0"}')
print(f'[OK] HADOOP_HOME  : {hadoop_home or "NOT FOUND — tải winutils.exe"}')
print()

if spark_submit:
    packages = (
        'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,'
        'org.mongodb.spark:mongo-spark-connector_2.12:10.3.0'
    )
    print('Chạy lệnh sau trong terminal riêng (giữ chạy):')
    print('-' * 72)
    if hadoop_home:
        print(f'set HADOOP_HOME={hadoop_home}')
        print(f'set PATH={hadoop_home}\\bin;%PATH%')
    print(f'set PYSPARK_PYTHON={python_exe}')
    print(f'set PYSPARK_DRIVER_PYTHON={python_exe}')
    print(f'{spark_submit} --packages "{packages}" src\\task5\\ingest.py')
    print('-' * 72)
else:
    print('[WARN] Không tìm thấy spark-submit — pip install pyspark==3.5.0')


[OK] Python       : c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\python.exe
[OK] spark-submit : C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Scripts\spark-submit.CMD
[OK] HADOOP_HOME  : NOT FOUND — tải winutils.exe

Chạy lệnh sau trong terminal riêng (giữ chạy):
------------------------------------------------------------------------
set PYSPARK_PYTHON=c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\python.exe
set PYSPARK_DRIVER_PYTHON=c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\python.exe
C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Scripts\spark-submit.CMD --packages "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.mongodb.spark:mongo-spark-connector_2.12:10.3.0" src\task5\ingest.py
------------------------------------------------------------------------


### 3. Kiểm chứng dữ liệu trong MongoDB

In [23]:
from pymongo import MongoClient
import json, time

MONGO_URI  = 'mongodb://127.0.0.1:27017'
DB_NAME    = 'peft_db'
COLL_NAME  = 'source_metadata'

print(f'[INFO] Kết nối MongoDB: {MONGO_URI}/{DB_NAME}.{COLL_NAME}')
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
coll   = client[DB_NAME][COLL_NAME]

count = coll.count_documents({})
print(f'[OK] Số documents trong collection: {count}')

if count > 0:
    sample = coll.find_one({}, {'_id': 0})
    print('\nDocument mẫu:')
    print(json.dumps(sample, indent=2, default=str))
else:
    print('[WARN] Chưa có dữ liệu. Đảm bảo spark-submit đang chạy và emit.py đã được chạy trước.')

client.close()

[INFO] Kết nối MongoDB: mongodb://127.0.0.1:27017/peft_db.source_metadata
[OK] Số documents trong collection: 0
[WARN] Chưa có dữ liệu. Đảm bảo spark-submit đang chạy và emit.py đã được chạy trước.


### 4. Reflection

**What worked**
- Spark Structured Streaming với trigger `processingTime=10s` hoạt động ổn định, đọc topic `cpg-metadata` và ghi vào MongoDB.
- Checkpoint đảm bảo fault-tolerance: khi khởi động lại Spark, offset được khôi phục và không xử lý lại message cũ.
- MongoDB Spark Connector v10.3.0 tương thích với Spark 3.5.0, giải quyết lỗi `RowEncoder` của phiên bản cũ hơn.

**What failed**
- `HADOOP_HOME` chưa được cấu hình trên Windows khiến Spark không khởi động được. Cần tải `winutils.exe` và `hadoop.dll` từ Hadoop 3.3.5.
- `mongo-spark-connector:10.2.1` không tương thích với Spark 3.5.0 do API `RowEncoder` đã thay đổi.

**How resolved**
- Tải `winutils.exe` + `hadoop.dll` về `D:\hadoop\bin\`, set `HADOOP_HOME=D:\hadoop` và copy `hadoop.dll` vào `C:\Windows\System32\`.
- Nâng cấp lên `mongo-spark-connector:10.3.0` là phiên bản chính thức hỗ trợ Spark 3.5.x.